In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch

import sys, os
sys.path.append(os.path.join(os.getcwd(), ".."))

from tqdm import tqdm
from ml_force import MorrisLecarCurrent, z_transform, VanDerPol

In [2]:
T = 5000
dt = 5e-2
t = np.arange(0, T, dt)
nt = t.size

In [3]:
supervisor = VanDerPol(T, dt, mu=0.3, tau=0.02).generate(transient_time=100)

supervisor = z_transform(supervisor.T)

In [ ]:
plt.plot(t, supervisor[:, 1])
plt.show()

In [15]:
N = 400
I = 70      # pA
gbar = 50
Q = 200
l = 1e-4
p_sparse = 1

device = torch.device('cuda')

model = MorrisLecarCurrent(supervisor=supervisor, dt=dt, T=T, N=N, BIAS=I, Q=Q, l=l, gbar=gbar, device=device, p_sparsity=p_sparse)

model.w /= torch.linalg.eigvalsh(model.w).abs().max() * 1.2

In [ ]:
clr = plt.pcolormesh(model.w.cpu().numpy())
plt.colorbar(clr)
plt.show()

## Stress Tests

In [30]:
model.dec = torch.ones(size=(model._N, 2), device=model.device, dtype=torch.float64) * 10

In [ ]:
v_trace = torch.zeros((nt, N), device=device, dtype=torch.float64)
s_trace = torch.zeros((nt, N), device=device, dtype=torch.float64)

model.__reinit__()

for i in tqdm(range(nt // 5)):
    model.euler_step()
    v_trace[i] = model.v.squeeze()
    s_trace[i] = model.s.squeeze()

In [ ]:
nrows = 10

neurons = np.random.choice(model._N, nrows, replace=False)

fig, ax = plt.subplots(figsize=(10, 10), nrows=nrows, ncols=2, sharex=True)
for i in range(nrows):
    ind = neurons[i]
    ax[i, 0].plot(t, v_trace[:, ind].cpu().numpy(), c='g', lw=1)
    ax[i, 0].set_ylabel(f'{ind}', fontsize=10)
    ax[i, 1].plot(t, s_trace[:, ind].cpu().numpy(), c='k', lw=1)
ax[0, 0].set_title('Voltage traces', fontsize=15)
ax[0, 1].set_title('Synaptic-Gating traces', fontsize=15)
plt.xlabel('Time (ms)', fontsize=10)
plt.show(fig)

## Echo State

In [13]:
def echo_state(model:MorrisLecarCurrent, T:float, dt:float, t_transient:float=200, 
               lr:float=0.01, n_epochs=5, l_ridge:float=0.01):
    t = np.arange(0, T, dt)
    nt = t.size
    dim = model.sup.shape[1]
    
    nt_transient = round(t_transient // dt)
    for epoch in range(n_epochs):
        print(f"Epoch {epoch}/{n_epochs}: \n")
        preds = torch.zeros(size=(nt, dim), device=model.device, dtype=torch.float64)
        states = torch.zeros(size=(nt, model._N), device=model.device, dtype=torch.float64)
        model.__reinit__()
        for _ in tqdm(range(nt_transient)):
            model.euler_step(closed_loop=False)
        
        for i in tqdm(range(nt)):
            model.euler_step()
            preds[i] = model.x_hat.squeeze()
            states[i] = model.s.squeeze()

        loss = torch.mean((preds - model.sup)**2, dim=0, keepdims=True)
        print(f"Loss={loss}")
        # error = preds - model.sup
        z = states - states.mean(axis=0, keepdim=True)
        Pinv = torch.linalg.inv(z.T @ z + l_ridge * torch.eye(model._N, device=model.device))
        W_new = Pinv @ z.T @ model.sup
        print(W_new.abs().max().item(), W_new.size())
        # update rule for MSE loss with Ridge regression
        model.dec = W_new
        
        plt.plot(t, model.sup[:, 0].cpu().numpy(), 'b-', label="y")
        plt.plot(t, preds[:, 0].cpu().numpy(), 'g-', label="y_hat")
        plt.legend(loc=0)
        plt.show()
        plt.close()
    return model, states, preds

In [ ]:
model_trained, states, preds = echo_state(model, T, dt, n_epochs=10, lr=.1, l_ridge=1e-5)

In [ ]:
dim = model.sup.shape[1]
preds = torch.zeros(size=(nt, dim), device=model.device, dtype=torch.float64)

for i in tqdm(range(nt)):
    model.euler_step()
    preds[i] = model.x_hat.ravel()

plt.plot(t, supervisor[:, 0])
plt.plot(t, preds[:, 0].cpu().numpy())
plt.show()